In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install -q dagshub mlflow
import mlflow
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
dagshub_token = user_secrets.get_secret("DAGSHUB_TOKEN")
dagshub_username = user_secrets.get_secret("DAGSHUB_USERNAME")

os.environ['MLFLOW_TRACKING_USERNAME'] = dagshub_username
os.environ['MLFLOW_TRACKING_PASSWORD'] = dagshub_token

mlflow.set_tracking_uri(f"https://dagshub.com/{dagshub_username}/ML_assignment2.mlflow")
mlflow.end_run()

print(f"Successfully connected to {dagshub_username}'s MLflow server!")

Successfully connected to njvar23's MLflow server!


## Data Loading & Cleaning

In [5]:
import gc

train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

for col in train_transaction.select_dtypes('float64').columns:
    train_transaction[col] = train_transaction[col].astype('float32')
for col in train_identity.select_dtypes('float64').columns:
    train_identity[col] = train_identity[col].astype('float32')

train = pd.merge(train_transaction, train_identity, on='TransactionID', how='left')

del train_transaction, train_identity
gc.collect()

print(f"Setup Complete. Final Dataset Shape: {train.shape}")
print(f"Memory Usage: {train.memory_usage().sum() / 1024**2:.2f} MB")

mlflow.end_run()
mlflow.set_experiment("LogisticRegression_Training")

with mlflow.start_run(run_name="LR_Cleaning"):
    null_percent = train.isnull().sum() / len(train)
    cols_to_drop = null_percent[null_percent > 0.90].index.tolist()
    train.drop(columns=cols_to_drop, inplace=True)

    mlflow.log_param("missing_threshold", 0.90)
    mlflow.log_metric("cols_dropped", len(cols_to_drop))
    print(f"Cleaning complete. Dropped {len(cols_to_drop)} columns.")

Setup Complete. Final Dataset Shape: (590540, 434)
Memory Usage: 1056.53 MB
Cleaning complete. Dropped 12 columns.
🏃 View run LR_Cleaning at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2/runs/656dacf49ed3406bb94bf9bbafdba077
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2


## Feature Engineering

In [6]:
mlflow.end_run()
with mlflow.start_run(run_name="LR_Feature_Engineering"):
    train['Transaction_hour'] = (train['TransactionDT'] // 3600) % 24
    train['Transaction_day'] = (train['TransactionDT'] // (3600 * 24)) % 7
    train['card1_amt_mean'] = train.groupby('card1')['TransactionAmt'].transform('mean')
    train['amt_to_mean_card1'] = train['TransactionAmt'] / train['card1_amt_mean']
    train.drop(columns=['card1_amt_mean'], inplace=True)

    mlflow.log_param("added_features", ["Transaction_hour", "Transaction_day", "amt_to_mean_card1"])
    mlflow.log_metric("total_features_after_eng", len(train.columns))
    print(f"Feature engineering complete. Total columns: {len(train.columns)}")

Feature engineering complete. Total columns: 425
🏃 View run LR_Feature_Engineering at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2/runs/d0e0e6d7448a49848495c6bf881494ed
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2


## Feature Selection

In [7]:
mlflow.end_run()
with mlflow.start_run(run_name="LR_Feature_Selection"):
    target = 'isFraud'
    drop_cols = [target, 'TransactionID', 'TransactionDT']

    X = train.drop(columns=drop_cols)
    y = train[target]

    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

    constant_cols = [col for col in X.columns if X[col].nunique() <= 1]
    X.drop(columns=constant_cols, inplace=True)

    mlflow.log_param("dropped_identifiers", str(drop_cols))
    mlflow.log_param("constant_cols_dropped", str(constant_cols))
    mlflow.log_metric("final_feature_count", X.shape[1])
    print(f"Final feature count: {X.shape[1]}")

from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

Final feature count: 422
🏃 View run LR_Feature_Selection at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2/runs/3f94285884274fd9af68a1c26b9b4a82
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2


## Training and Model Pipeline

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score

def build_lr_preprocessor(num_cols, cat_cols):
    num_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ])
    cat_transformer = Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ('scaler', StandardScaler())
    ])
    return ColumnTransformer(transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ])

experiments = [
    {
        "run_name": "LR_baseline",
        "model": LogisticRegression(
            max_iter=1000,
            random_state=42,
            n_jobs=-1
        )
    },
    {
        "run_name": "LR_balanced",
        "model": LogisticRegression(
            max_iter=1000,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
    },
    {
        "run_name": "LR_strong_regularization",
        "model": LogisticRegression(
            C=0.01,
            max_iter=1000,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
    },
    {
        "run_name": "LR_weak_regularization",
        "model": LogisticRegression(
            C=10.0,
            max_iter=1000,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
    },
    {
        "run_name": "LR_ridge",  
        "model": LogisticRegression(
            penalty='l2',
            C=1.0,
            solver='lbfgs',
            max_iter=1000,
            class_weight='balanced',
            random_state=42,
            n_jobs=-1
        )
    },
]

for exp in experiments:
    mlflow.end_run()
    with mlflow.start_run(run_name=exp["run_name"]):
        clf = Pipeline(steps=[
            ('preprocessor', build_lr_preprocessor(num_cols, cat_cols)),
            ('model', exp["model"])
        ])

        clf.fit(X_train, y_train)
        val_preds = clf.predict_proba(X_val)[:, 1]
        auc_score = roc_auc_score(y_val, val_preds)

        mlflow.log_params(exp["model"].get_params())
        mlflow.log_metric("val_auc", auc_score)
        mlflow.sklearn.log_model(clf, "model", registered_model_name=exp["run_name"])

        print(f"[{exp['run_name']}] Validation AUC: {auc_score:.4f}")

2026/05/02 14:02:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 14:02:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LR_baseline' already exists. Creating a new version of this model...
2026/05/02 14:02:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LR_baseline, version 2
Created version '2' of model 'LR_baseline'.


[LR_baseline] Validation AUC: 0.8559
🏃 View run LR_baseline at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2/runs/baf2f89f23b1476585d8908c2792440d
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2


2026/05/02 14:06:00 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 14:06:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LR_balanced' already exists. Creating a new version of this model...
2026/05/02 14:06:07 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LR_balanced, version 2
Created version '2' of model 'LR_balanced'.


[LR_balanced] Validation AUC: 0.8629
🏃 View run LR_balanced at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2/runs/92e34fa9b4d348d79c5bdf5b0335f3bc
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2


2026/05/02 14:08:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 14:08:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LR_strong_regularization' already exists. Creating a new version of this model...
2026/05/02 14:08:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LR_strong_regularization, version 2
Created version '2' of model 'LR_strong_regularization'.


[LR_strong_regularization] Validation AUC: 0.8620
🏃 View run LR_strong_regularization at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2/runs/3786c8eeead4475095c45a0708b06d98
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2


2026/05/02 14:11:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 14:11:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'LR_weak_regularization' already exists. Creating a new version of this model...
2026/05/02 14:11:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LR_weak_regularization, version 2
Created version '2' of model 'LR_weak_regularization'.


[LR_weak_regularization] Validation AUC: 0.8629
🏃 View run LR_weak_regularization at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2/runs/031f732a167f41b1905be5bf34b70172
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2


2026/05/02 14:15:22 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/02 14:15:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LR_ridge'.
2026/05/02 14:15:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LR_ridge, version 1
Created version '1' of model 'LR_ridge'.


[LR_ridge] Validation AUC: 0.8629
🏃 View run LR_ridge at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2/runs/933a4d0ad9ee43318717d45c8fb8a6f1
🧪 View experiment at: https://dagshub.com/njvar23/ML_assignment2.mlflow/#/experiments/2
